In [6]:
import os

import pandas as pd
import torch
from transformers import pipeline
from tqdm import tqdm

In [7]:
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = 0
else:
    device = -1

In [8]:
classifier = pipeline(
    "text-classification",
    model="monologg/bert-base-cased-goemotions-original",
    top_k=None,
    device=device,
    token=os.getenv("HF_TOKEN"),
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [9]:
df = pd.read_csv("../../../data/cleaned_data.csv")

print("Rozpoczynam klasyfikację...")
texts = df['cleaned_statement'].astype(str).tolist()

Rozpoczynam klasyfikację...


In [ ]:
results = []
for out in tqdm(classifier(texts, batch_size=16, truncation=True), total=len(texts)):
    results.append(out)

In [ ]:
extracted_data = []
for res in results:
    row = {item['label']: item['score'] for item in res}
    extracted_data.append(row)

In [ ]:
features_df = pd.DataFrame(extracted_data)

In [ ]:
if 'label' in df.columns:
    features_df['target_label'] = df['label'].values

features_df.to_csv('../../../data/features_for_model.csv', index=False)